# 03 — Enrichment (Obtain, part 2)

Adds biographical data to the riders table by querying **Wikidata** over SPARQL.

**Why an API rather than more scraping.** The original version of this project scraped each
rider's individual Wikipedia page looking for `<span class="bday">` in the infobox — roughly
100 HTTP requests, each parsing rendered HTML whose structure varies by page.

Wikidata holds the same facts as structured data. Every entity is a set of subject–property–object
triples: *Eddy Merckx (Q103756) → date of birth (P569) → 1945-06-17*. One query returns all 164
riders at once.

| Property | Meaning |
|---|---|
| `P106` | occupation (filtered to `Q2309784`, cyclist) |
| `P569` | date of birth |
| `P27` | country of citizenship |
| `P2048` | height |
| `P2067` | mass |

**Why the occupation filter matters.** Querying `rdfs:label "Eddy Merckx"@en` without it returns
two entities — the cyclist (b. 1945) and an unrelated person of the same name (b. 1968).
Constraining on `P106 = cyclist` is the primary defence against wrong matches.

In [31]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

from src.wikidata import fetch_riders

riders = pd.read_csv("../data/processed/riders.csv")
print(riders.shape)
riders.head()

(161, 7)


,rider_name,country,wins_tdf,wins_giro,wins_vuelta,wins_total,races_won
0,Abraham Olano,Spain,0,0,1,1,1
1,Agustín Tamames,Spain,0,0,1,1,1
2,Aitor González,Spain,0,0,1,1,1
3,Alberto Contador,Spain,2,2,3,7,3
4,Alejandro Valverde,Spain,0,0,1,1,1


In [32]:
names = riders["rider_name"].dropna().unique().tolist()
print(f"Querying Wikidata for {len(names)} riders...")

raw_wd = fetch_riders(names, batch_size=50)
wd = pd.DataFrame(raw_wd)
print("\nRows returned:", len(wd))
wd.head()

Querying Wikidata for 161 riders...
batch 1: 50 names -> 63 rows
batch 2: 50 names -> 61 rows
batch 3: 50 names -> 58 rows
batch 4: 11 names -> 12 rows

Rows returned: 194


,query_name,wikidata_label,birth_date,country_wikidata,height_cm,weight_kg
0,Denis Menchov,Denis Menshov,1978-01-25T00:00:00Z,Russia,180.0,65.0
1,Aitor González,Aitor González Jiménez,1975-02-27T00:00:00Z,Spain,177.0,NaN
2,Egan Bernal,Egan Bernal,1997-01-13T00:00:00Z,Colombia,175.0,60.0
3,Aitor González,Aitor González Prieto,1990-11-05T00:00:00Z,Spain,NaN,NaN
4,Alexander Vinokourov,Alexandre Vinokourov,2002-07-07T00:00:00Z,Kazakhstan,NaN,NaN


## Handling duplicate rows

Wikidata returns one row per citizenship, so dual-national riders appear more than once
(Chris Froome: Kenya and United Kingdom). Rather than discarding one, the citizenships are
collapsed into a single comma-separated field — Froome being Kenyan-born and British-licensed
is a genuine fact worth keeping.

In [33]:
wd_grouped = (wd.groupby("query_name")
                .agg(birth_date=("birth_date", "first"),
                     citizenships=("country_wikidata", lambda s: ", ".join(sorted(set(s.dropna())))),
                     height_cm=("height_cm", "first"),
                     weight_kg=("weight_kg", "first"))
                .reset_index())

print("Unique riders matched:", len(wd_grouped))
print("\nDual/multiple citizenship:")
print(wd_grouped[wd_grouped["citizenships"].str.contains(",", na=False)][
    ["query_name", "citizenships"]].to_string(index=False))

Unique riders matched: 158

Dual/multiple citizenship:
         query_name                         citizenships
  Alfonso Calzolari              Italy, Kingdom of Italy
      Alfredo Binda              Italy, Kingdom of Italy
    Angelo Conterno              Italy, Kingdom of Italy
    Antonio Pesenti              Italy, Kingdom of Italy
      Carlo Clerici Italy, Kingdom of Italy, Switzerland
      Carlo Galetti              Italy, Kingdom of Italy
       Chris Froome                Kenya, United Kingdom
Costante Girardengo              Italy, Kingdom of Italy
      Evgeni Berzin                 Russia, Soviet Union
       Fausto Coppi              Italy, Kingdom of Italy
     Fiorenzo Magni              Italy, Kingdom of Italy
  Francesco Camusso              Italy, Kingdom of Italy
     François Faber                   France, Luxembourg
    Gaetano Belloni              Italy, Kingdom of Italy
    Gastone Nencini              Italy, Kingdom of Italy
       Gino Bartali              

In [34]:
matched = set(wd_grouped["query_name"])
unmatched = [n for n in names if n not in matched]

print(f"Matched   : {len(matched)}/{len(names)} ({len(matched)/len(names):.1%})")
print(f"Unmatched : {len(unmatched)}")
print("\nUnmatched riders:")
for n in unmatched:
    print(" -", n)

Matched   : 158/161 (98.1%)
Unmatched : 3

Unmatched riders:
 - Antonio Suárez
 - José Pesarrodona
 - Miguel Indurain


In [35]:
# A name matching multiple distinct birth dates means it matched multiple people
ambiguity = wd.groupby("query_name")["birth_date"].nunique()
ambiguous = ambiguity[ambiguity > 1].index.tolist()

print("Names matching multiple people (excluded):", ambiguous)
wd_grouped = wd_grouped[~wd_grouped["query_name"].isin(ambiguous)]
print("Riders retained:", len(wd_grouped))

Names matching multiple people (excluded): ['Aitor González', 'Alexander Vinokourov']
Riders retained: 156


## Match results

| Outcome | Count |
|---|---|
| Matched unambiguously | 156 |
| Matched multiple people (excluded) | 2 |
| No match | 5 |
| **Total riders** | **161** |

**Why matching is imperfect.** Wikidata is queried by label, and labels do not always match
Wikipedia's article titles:

- **Transliteration** — Wikipedia writes *Denis Menchov*; Wikidata's label is *Denis Menshov*,
  the more accurate romanisation of Меньшов. Solved by also matching `skos:altLabel`, which
  holds alternative names for each entity.
- **Accents** — *Miguel Indurain* vs *Miguel Induráin*. Not resolved by alt-labels in this case.
- **Ambiguity** — *Aitor González* matches two different Spanish cyclists (b. 1975 and b. 1990);
  *Alexander Vinokourov* likewise. The occupation filter cannot separate them because both
  candidates are cyclists.

**Ambiguous names are excluded, not guessed.** A name returning two distinct birth dates has
matched two people. Picking one arbitrarily would insert a plausible-looking wrong value —
worse than a missing one, because it cannot be detected downstream. Note that the ambiguity
check itself surfaced Vinokourov, who had previously been counted as a successful match.

The unmatched riders are each one-time Grand Tour winners, so their absence has limited effect
on the analysis. Every affected chart states its coverage.

In [36]:
riders_enriched = riders.merge(
    wd_grouped, left_on="rider_name", right_on="query_name", how="left"
).drop(columns=["query_name"])

riders_enriched["birth_date"] = pd.to_datetime(
    riders_enriched["birth_date"], format="mixed", utc=True
).dt.tz_localize(None)

print("Riders with a birth date:", riders_enriched["birth_date"].notna().sum(), "/", len(riders_enriched))
riders_enriched.head()

Riders with a birth date: 156 / 161


,rider_name,country,wins_tdf,wins_giro,wins_vuelta,wins_total,races_won,birth_date,citizenships,height_cm,weight_kg
0,Abraham Olano,Spain,0,0,1,1,1,1970-01-22,Spain,181.0,70.0
1,Agustín Tamames,Spain,0,0,1,1,1,1944-10-19,Spain,173.0,66.0
2,Aitor González,Spain,0,0,1,1,1,NaT,NaN,NaN,NaN
3,Alberto Contador,Spain,2,2,3,7,3,1982-12-06,Spain,176.0,62.0
4,Alejandro Valverde,Spain,0,0,1,1,1,1980-04-25,Spain,177.0,61.0


In [37]:
editions = pd.read_csv("../data/processed/editions.csv")

ed = editions.merge(
    riders_enriched[["rider_name", "birth_date", "height_cm", "weight_kg", "citizenships"]],
    on="rider_name", how="left",
)

# The Tour finishes in July, the Giro in late May/early June, the Vuelta in September.
RACE_END = {"tdf": "07-25", "giro": "06-01", "vuelta": "09-15"}

race_end_date = pd.to_datetime(
    ed["year"].astype(str) + "-" + ed["race"].map(RACE_END)
)

ed["age_at_win"] = (race_end_date - ed["birth_date"]).dt.days / 365.25

print("Editions with a computable age:", ed["age_at_win"].notna().sum())
print(ed["age_at_win"].describe().round(2))

Editions with a computable age: 288
count    288.00
mean      27.92
std        3.33
min       19.97
25%       25.58
50%       27.74
75%       30.18
max       41.90
Name: age_at_win, dtype: float64


In [38]:
print(ed.nlargest(5, "age_at_win")[
    ["race", "year", "rider_name", "age_at_win", "birth_date"]
].to_string(index=False))

print()
print(ed.nsmallest(5, "age_at_win")[
    ["race", "year", "rider_name", "age_at_win", "birth_date"]
].to_string(index=False))

  race  year      rider_name  age_at_win birth_date
vuelta  2013    Chris Horner   41.897331 1971-10-23
   tdf  1922   Firmin Lambot   36.361396 1886-03-14
vuelta  2024   Primož Roglič   34.880219 1989-10-29
   tdf  1923 Henri Pélissier   34.499658 1889-01-22
  giro  1955  Fiorenzo Magni   34.480493 1920-12-07

  race  year       rider_name  age_at_win birth_date
   tdf  1904     Henri Cornet   19.969884 1884-08-04
  giro  1940     Fausto Coppi   20.711841 1919-09-15
  giro  1930  Luigi Marchisio   21.097878 1909-04-26
  giro  1979 Giuseppe Saronni   21.689254 1957-09-22
vuelta  1961   Angelino Soler   21.806982 1939-11-25


In [39]:
checks = {
    "ages are plausible (19-45)":
        ed["age_at_win"].dropna().between(19, 45).all(),
    "heights are plausible (150-210cm)":
        riders_enriched["height_cm"].dropna().between(150, 210).all(),
    "weights are plausible (50-100kg)":
        riders_enriched["weight_kg"].dropna().between(50, 100).all(),
    "no rider matched two birth dates":
        riders_enriched.groupby("rider_name")["birth_date"].nunique().max() <= 1,
    "editions row count unchanged": len(ed) == 333,
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

PASS  ages are plausible (19-45)
PASS  heights are plausible (150-210cm)
PASS  weights are plausible (50-100kg)
PASS  no rider matched two birth dates
PASS  editions row count unchanged


In [40]:
riders_enriched.to_csv("../data/processed/riders_enriched.csv", index=False)
ed.to_csv("../data/processed/editions_enriched.csv", index=False)
print("riders_enriched :", riders_enriched.shape)
print("editions_enriched:", ed.shape)

riders_enriched : (161, 11)
editions_enriched: (333, 18)
